Fetch form definitions


In [1]:
import {
  REGISTRAR_USERNAME,
  REGISTRAR_PASSWORD
} from '../commonHelpers/vars.ts'
import { authenticate } from '../commonHelpers/authentication.ts'

const token = await authenticate(REGISTRAR_USERNAME, REGISTRAR_PASSWORD)
token

"eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJzY29wZSI6WyJyZWNvcmQucmVhZCIsInJlY29yZC5kZWNsYXJlLWJpcnRoIiwicmVjb3JkLmRlY2xhcmUtZGVhdGgiLCJyZWNvcmQuZGVjbGFyZS1tYXJyaWFnZSIsInJlY29yZC5kZWNsYXJhdGlvbi1zdWJtaXQtZm9yLXVwZGF0ZXMiLCJyZWNvcmQudW5hc3NpZ24tb3RoZXJzIiwicmVjb3JkLmRlY2xhcmF0aW9uLWVkaXQiLCJyZWNvcmQuZGVjbGFyYXRpb24tYXJjaGl2ZSIsInJlY29yZC5kZWNsYXJhdGlvbi1yZWluc3RhdGUiLCJyZWNvcmQucmV2aWV3LWR1cGxpY2F0ZXMiLCJyZWNvcmQucmVnaXN0ZXIiLCJyZWNvcmQucmVnaXN0cmF0aW9uLXByaW50Jmlzc3VlLWNlcnRpZmllZC1jb3BpZXMiLCJwcm9maWxlLmVsZWN0cm9uaWMtc2lnbmF0dXJlIiwicmVjb3JkLnJlZ2lzdHJhdGlvbi1jb3JyZWN0IiwicmVjb3JkLmNvbmZpcm0tcmVnaXN0cmF0aW9uIiwicmVjb3JkLnJlamVjdC1yZWdpc3RyYXRpb24iLCJwZXJmb3JtYW5jZS5yZWFkIiwicGVyZm9ybWFuY2UucmVhZC1kYXNoYm9hcmRzIiwicHJvZmlsZS5lbGVjdHJvbmljLXNpZ25hdHVyZSIsInVzZXIucmVhZDpvbmx5LW15LWF1ZGl0Iiwib3JnYW5pc2F0aW9uLnJlYWQtbG9jYXRpb25zOm15LW9mZmljZSIsInBlcmZvcm1hbmNlLnZpdGFsLXN0YXRpc3RpY3MtZXhwb3J0Iiwic2VhcmNoLmJpcnRoIiwic2VhcmNoLmRlYXRoIiwic2VhcmNoLm1hcnJpYWdlIiwib3JnYW5pc2F0aW9uLnJlYWQtbG9jYXRpb2

In [2]:
import { extractFieldType, Field } from '../commonHelpers/utils.ts'
import { fetchEvents } from '../commonHelpers/formsHandlers.ts'

const ignoredFields = [
  'DIVIDER',
  'PARAGRAPH',
  'BULLET_LIST',
  'HEADING3',
  'SIGNATURE',
  'PAGE_HEADER',
  'FILE',
  'FILE_WITH_OPTIONS',
  'SEARCH',
  'VERIFICATION_STATUS',
  'QUERY_PARAM_READER',
  'HTTP',
  'LOADER',
  'ID_READER',
  'collector.*',
  'requester.*',
  'review.*',
  'fees.*',
  'lateFee.*',
  'documents.*',
  'informant.contact',
  'reason.option',
  'reason.other'
]

const events = await fetchEvents(token)

type StreetAddressField = { id: string; type: string; addressType: 'DOMESTIC' | 'INTERNATIONAL' }
type EventFieldDescriptor = {
  id: string
  type: string
  options?: string[]
  nameFields?: string[]
  streetAddressFields?: StreetAddressField[]
  defaultCountry?: string
}

const eventDescription = events.reduce(
  (allEvents, event) => {
    const uniqueFields = new Map<string, EventFieldDescriptor>()

    extractFieldType(event, 'fields')
      .filter((x) => !ignoredFields.includes(x.type))
      .filter((x) => x.id)
      .filter(
        (x) => !ignoredFields.some((pattern) => new RegExp(pattern).test(x.id))
      )
      .forEach((f) => {
        uniqueFields.set(f.id, {
          id: f.id,
          type: f.type,
          options: f.options?.map((o) => o.value),
          ...(f.type === 'NAME' && f.configuration?.name
            ? { nameFields: Object.keys(f.configuration.name as Record<string, unknown>) }
            : {}),
          ...(f.type === 'ADDRESS' && f.configuration?.streetAddressForm
            ? {
                streetAddressFields: (f.configuration.streetAddressForm as { id: string; type: string; conditionals?: unknown }[]).map((s) => ({
                  id: s.id,
                  type: s.type,
                  addressType: JSON.stringify(s.conditionals).includes('DOMESTIC') ? 'DOMESTIC' : 'INTERNATIONAL'
                })),
                ...((f.defaultValue as Record<string, unknown>)?.country
                  ? { defaultCountry: (f.defaultValue as Record<string, unknown>).country as string }
                  : {})
              }
            : {}),
          ...(f.type === 'COUNTRY' && typeof f.defaultValue === 'string'
            ? { defaultCountry: f.defaultValue }
            : {})
        })
      })

    allEvents[event.id] = [...uniqueFields.values()]

    return allEvents
  },
  {} as Record<string, EventFieldDescriptor[]>
)

Deno.writeFileSync(
  './formData/eventDescription.json',
  new TextEncoder().encode(JSON.stringify(eventDescription, null, 2))
)

Deno.writeFileSync(
  './formData/fullEvent.json',
  new TextEncoder().encode(JSON.stringify(events, null, 2))
)

In [3]:
import { generateTypes } from './helpers/typeGenerator.ts'

generateTypes(eventDescription)

Written: /home/baz/code/openCRVS/notebooks/data-generators/types/sharedTypes_generated.ts
Written: /home/baz/code/openCRVS/notebooks/data-generators/types/tennisClubMembershipTypes_generated.ts
Written: /home/baz/code/openCRVS/notebooks/data-generators/types/birthTypes_generated.ts
Written: /home/baz/code/openCRVS/notebooks/data-generators/types/deathTypes_generated.ts
